# 回归模型的 LM 检验模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.13/simulation_lm_ols_test.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对简单回归构造 LM 统计量，模拟 β=0（H0）与 β=0.2（H1）下的分布与拒绝率。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空数据、关闭分页、固定种子。本 notebook 考察回归模型 y=α+βx+ε 中 **H0: β=0 的 LM 检验**。


In [ ]:
%%stata
clear
set more off
set seed 42


**test statistic**

定义统计量程序 `ols_lm_test_stat`：生成 x~N(0,4)、y=α+βx+ε 后，先在**原假设**（β=0）下估计并求残差：`e=y-x̄`（只需常数项的模型）；再构造乘积序列 `xe=x·e`，返回 `chi2_stat=(Σxe)²/(n·Var(xe))`。
**推导思路**：检验 β=0 的得分统计量基于 Σx_i·e_i（x 与残差正交）。程序把它写成 `n·(mean(xe))²/Var(xe)` 的形式——即“乘积序列 xe 的均值是否为零”的 t 统计量平方；等价于辅助回归的 n·R² 形式，在原假设下渐近服从 χ²(1)。


In [ ]:
%%stata
cap program drop ols_lm_test_stat
program define ols_lm_test_stat, rclass
	syntax [,obs(integer 100) beta(real 0) alpha(real 1) sigma(real 1)]
	clear
	tempvar x y e xe
	quietly{
		set obs `obs'
		gen `x'=rnormal(0,2)
		gen `y'=`x'*`beta'+`alpha'+rnormal(0,`sigma')
		su `y'
		gen `e'=`y'-r(mean)
		gen `xe'=`x'*`e'
		sum `xe'
		return scalar chi2_stat=(r(sum)^2)/(`obs'*r(Var))
	}
end


**upper tail of chi2(1)**

计算 χ²(1) 的95%临界值。


In [ ]:
%%stata
local cv=invchi2tail(1,0.05)


**simulation under H0**

**H0 下的模拟**：默认 β=0（原假设成立）重复5000次，统计量直方图与 χ²(1) 密度、临界值对比；`su p_005` 给出实际拒绝率（经验证约5.5%，与名义水平吻合）。


In [ ]:
%%stata
simulate chi2=r(chi2_stat),reps(5000): ols_lm_test_stat
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005


**simulation under H1**

**H1 下的模拟**：以 β=0.2（弱备择假设）重复5000次——拒绝率高于5%但不会很高，因为效应较弱；这提示 LM 检验的功效依赖于备择假设离原假设的“距离”。


In [ ]:
%%stata
clear
simulate chi2=r(chi2_stat),reps(5000): ols_lm_test_stat, beta(0.2)
gen density=chi2den(1,chi2) if chi2>0.1
sort chi2
twoway (hist chi2, density) (line density chi2), xline(`cv')
gen p_005=chi2>=`cv'
su p_005
